# 04 — Unsupervised Layer (Incident Motion Clustering)
**RoadSentinel AI** — Clusters flagged traffic incidents by motion dynamics (speed, deceleration, IOU overlap).
Validates cluster count $k$ using Silhouette score analysis.

In [ ]:
import os, sys
sys.path.insert(0, '..')
import pandas as pd, joblib, matplotlib.pyplot as plt
from src.clustering import find_best_k, train_severity_kmeans

## 1. Filter Flagged Incidents for Categorization
Only incidents flagged by triage (or known accidents) are grouped into severity clusters.

In [ ]:
df = pd.read_csv('../data/engineered_features.csv')
num_cols = ['avg_speed', 'max_speed', 'max_deceleration', 'trajectory_variance', 'max_iou']
accidents = df[df['is_accident'] == 1][num_cols]
print(f"Analyzing {len(accidents)} flagged incidents for severity clustering.")

## 2. Silhouette Score Optimization (Why k?)
Computes silhouette coefficient across $k \in [2, 6]$ to justify chosen cluster count.

In [ ]:
best_k, scores = find_best_k(accidents, k_range=range(2, 6), save_plot_path='../models/metrics/silhouette_curve.png')
print(f"Optimal cluster count k = {best_k}")

## 3. Fit K-Means & Inspect Cluster Centroids

In [ ]:
kmeans_model = train_severity_kmeans(accidents, k=best_k)
centroids_df = pd.DataFrame(kmeans_model.cluster_centers_, columns=num_cols)
print("Cluster Centroids:")
centroids_df

## 4. Save Severity K-Means Artifact

In [ ]:
joblib.dump(kmeans_model, '../models/severity_kmeans.joblib')
print("Saved ../models/severity_kmeans.joblib")